# Getting started — Generalized Superposition Theorem (GST)

This notebook mirrors `getting_started_sup_theorem.ipynb`, but for the **Generalized
Superposition Theorem**, which combines **topological changes** *and* an **injection
change** (e.g. a redispatch) in a single linear, solver-free reconstruction.

**Available inputs** (exactly what an operator precomputes once):
* the reference state $\mathrm{PF}(P^{ref}, T^{ref})$,
* the unitary **topology** states $\mathrm{PF}(P^{ref}, T^{\tau_i})$ (one line/sub action each, reference injection),
* the unitary **injection** states $\mathrm{PF}(\delta P^k, T^{ref})$ (one injection change each, **reference topology**).

The modified-topology injection response $\mathrm{PF}(\delta P^k, T^{\tau_i})$ is **never** needed.

**Result.**
$$\mathrm{PF}(P^{tgt},T^{tgt}) = \alpha\, \mathrm{PF}(P^{ref},T^{ref})
 + \sum_i \beta^i\, \mathrm{PF}(P^{ref},T^{\tau_i})
 + \sum_k \mathrm{PF}(\delta P^k,T^{ref}),\qquad \alpha = 1-\sum_i\beta^i.$$

The topology states carry the **EST weights** $(\alpha,\beta^i)$; the injection states are added in **pure superposition**. The injection enters the weights **only through the right-hand side** of the (unchanged) EST linear system, as a flow/angle ratio on the switched assets:
$$\beta^l + \sum_{i\neq l}\beta^i\, tsf^{\tau_i}_l = \frac{x_l(P^{tgt},T^{ref})}{x_l(P^{ref},T^{ref})}.$$
With no injection change the right-hand side is $1$ and we recover the EST exactly.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import grid2op
from grid2op.Parameters import Parameters
from lightsim2grid import LightSimBackend
from superposition_theorem import State
from superposition_theorem.core.compute_power_flows import (
    compute_flows_GST_from_unit_act_obs,
    compute_flows_superposition_theorem_from_unit_act_obs,
)

param = Parameters()
param.ENV_DC = True                      # DC power flow, where the theorem is exact
param.MAX_LINE_STATUS_CHANGED = 99999
param.MAX_SUB_CHANGED = 99999
param.NO_OVERFLOW_DISCONNECTION = True
env = grid2op.make("l2rpn_case14_sandbox", param=param, backend=LightSimBackend(), test=True)
A = env.action_space
decimal_digit_precision = 4              # same accuracy check as the EST notebook

## The injection lever

In Grid2Op the injection change is a **balanced redispatch** (it sums to zero, so it
is applied verbatim and superposes linearly under DC). The unitary injection state is
just `obs_start.simulate(redispatch, 0)`, and its flow response is
`e = obs_inj.p_or - obs_start.p_or`.

## Example A — two line disconnections + an injection change

In [2]:
env.set_id(0); env.reset()
obs_start, *_ = env.step(A({}))                 # reference state (all lines in service)

id_l1, id_l2 = 3, 12
redisp = [(0, +5.0), (5, -5.0)]                 # balanced injection change dP

# unitary TOPOLOGY states (one disconnection each, reference injection)
obs1, *_ = obs_start.simulate(A({"set_line_status": [(id_l1, -1)]}), 0)
obs2, *_ = obs_start.simulate(A({"set_line_status": [(id_l2, -1)]}), 0)
# unitary INJECTION state (redispatch alone, reference topology)
obs_inj, *_ = obs_start.simulate(A({"redispatch": redisp}), 0)

# ground truth: the two disconnections AND the redispatch, together
combined = A({"set_line_status": [(id_l1, -1), (id_l2, -1)], "redispatch": redisp})
obs_target, *_ = obs_start.simulate(combined, 0)

# --- GST reconstruction (functional API) ---
p_gst, alpha, betas = compute_flows_GST_from_unit_act_obs(
    [id_l1, id_l2], obs_start, [obs1, obs2], [obs_inj], return_betas=True)
print("alpha =", round(alpha, 4), " betas =", np.round(betas, 4))
print(f"Max difference (MW): {np.max(np.abs(obs_target.p_or - p_gst))}")
assert np.all(np.round(obs_target.p_or - p_gst, decimal_digit_precision) == 0.0)

alpha = -1.1185  betas = [1.0263 1.0922]
Max difference (MW): 4.751127079316575e-06


The same result through the **object-oriented `State` API** (parallel to
`compute_flows_disco_lines`):

In [3]:
st = State.from_grid2op_obs(obs_start, line_ids_disc_unary=[id_l1, id_l2])
st.add_injection_states_grid2op(obs_start, [A({"redispatch": redisp})])
p_gst_oo = st.compute_flows_gst(line_ids_disc=[id_l1, id_l2])
print(f"Max difference (MW): {np.max(np.abs(obs_target.p_or - p_gst_oo))}")
assert np.all(np.round(obs_target.p_or - p_gst_oo, decimal_digit_precision) == 0.0)

Max difference (MW): 4.751127079316575e-06


**Why the injection cannot be bolted on naively.** Two tempting shortcuts are
wrong: (i) using the EST for the topology and *ignoring* the injection, and (ii) the
"separable" estimate = EST result at the reference injection **plus** the injection
response. Both miss the injection's effect on the switched-line flows (which the GST
puts in the right-hand side).

In [4]:
# (i) EST only, ignoring the injection
p_est_only = compute_flows_superposition_theorem_from_unit_act_obs(
    [id_l1, id_l2], [], obs_start, obs_target, [obs1, obs2])
print(f"EST only (ignores injection)      Max difference (MW): {np.max(np.abs(obs_target.p_or - p_est_only)):.4f}")

# (ii) naive separable: EST(ref) + injection response
p_naive = p_est_only + (obs_inj.p_or - obs_start.p_or)
print(f"naive separable (EST_ref + dP)    Max difference (MW): {np.max(np.abs(obs_target.p_or - p_naive)):.4f}")

print(f"GST (correct)                     Max difference (MW): {np.max(np.abs(obs_target.p_or - p_gst)):.2e}")

EST only (ignores injection)      Max difference (MW): 4.2682
naive separable (EST_ref + dP)    Max difference (MW): 0.2862
GST (correct)                     Max difference (MW): 4.75e-06


## Example B — two line reconnections + an injection change

Reconnections use the angle differences $\Delta\theta$ (the reconnected lines carry no
flow at the reference topology). The GST right-hand side becomes the **angle ratio**
$\Delta\theta_l(P^{tgt},T^{ref})/\Delta\theta_l(P^{ref},T^{ref})$, handled automatically.

In [5]:
env.set_id(0); env.reset()
# reference topology: the two lines are disconnected (to be reconnected)
obs_start, *_ = env.step(A({"set_line_status": [(id_l1, -1), (id_l2, -1)]}))

obs1, *_ = obs_start.simulate(A({"set_line_status": [(id_l1, +1)]}), 0)
obs2, *_ = obs_start.simulate(A({"set_line_status": [(id_l2, +1)]}), 0)
obs_inj, *_ = obs_start.simulate(A({"redispatch": redisp}), 0)

combined = A({"set_line_status": [(id_l1, +1), (id_l2, +1)], "redispatch": redisp})
obs_target, *_ = obs_start.simulate(combined, 0)

p_gst, alpha, betas = compute_flows_GST_from_unit_act_obs(
    [id_l1, id_l2], obs_start, [obs1, obs2], [obs_inj], return_betas=True)
print("alpha =", round(alpha, 4), " betas =", np.round(betas, 4))
print(f"GST       Max difference (MW): {np.max(np.abs(obs_target.p_or - p_gst))}")
p_est_only = compute_flows_superposition_theorem_from_unit_act_obs(
    [id_l1, id_l2], [], obs_start, obs_target, [obs1, obs2])
print(f"EST only  Max difference (MW): {np.max(np.abs(obs_target.p_or - p_est_only)):.4f}")
assert np.all(np.round(obs_target.p_or - p_gst, decimal_digit_precision) == 0.0)

alpha = -0.9064  betas = [0.9915 0.9149]
GST       Max difference (MW): 5.0188046785137885e-06
EST only  Max difference (MW): 4.1901


## Example C — two injection changes ($n_P=2$) + two line disconnections

Several injection changes simply add (standard superposition). Here the combined
redispatch equals $\delta P^1 + \delta P^2$ nodally.

In [6]:
env.set_id(0); env.reset()
obs_start, *_ = env.step(A({}))
dP1 = [(0, +5.0), (5, -5.0)]
dP2 = [(1, +5.0), (5, -5.0)]

obs1, *_ = obs_start.simulate(A({"set_line_status": [(id_l1, -1)]}), 0)
obs2, *_ = obs_start.simulate(A({"set_line_status": [(id_l2, -1)]}), 0)
obs_i1, *_ = obs_start.simulate(A({"redispatch": dP1}), 0)
obs_i2, *_ = obs_start.simulate(A({"redispatch": dP2}), 0)

combined = A({"set_line_status": [(id_l1, -1), (id_l2, -1)],
              "redispatch": [(0, +5.0), (1, +5.0), (5, -10.0)]})   # = dP1 + dP2
obs_target, *_ = obs_start.simulate(combined, 0)

p_gst = compute_flows_GST_from_unit_act_obs(
    [id_l1, id_l2], obs_start, [obs1, obs2], [obs_i1, obs_i2])
print(f"GST (2 injections + 2 disconnections)  Max difference (MW): {np.max(np.abs(obs_target.p_or - p_gst))}")
assert np.all(np.round(obs_target.p_or - p_gst, decimal_digit_precision) == 0.0)

GST (2 injections + 2 disconnections)  Max difference (MW): 3.7077106682659178e-06


## Substation actions (node split / merge) + injection

A substation reconfiguration is handled through its **virtual line** (exactly as in
the EST): a **node split** behaves like a line *disconnection* (virtual-flow ratio),
a **node merge** like a line *reconnection* (virtual-angle ratio). The injection
shifts these virtual sensitivities just like real-line ones. Pass the substation ids
through `idls_subs`, with `unit_act_observations` ordered **lines first, then subs**.

In [7]:
# Example D — two node splits + an injection change
env.set_id(0); env.reset(); obs_start, *_ = env.step(A({}))
id_sub1, id_sub2 = 5, 4
act_s1 = {"set_bus": {"substations_id": [(5, (1, 1, 2, 2, 1, 2, 2))]}}
act_s2 = {"set_bus": {"substations_id": [(4, (2, 1, 2, 1, 2))]}}
obs1, *_ = obs_start.simulate(A(act_s1), 0)                 # substation unitary states
obs2, *_ = obs_start.simulate(A(act_s2), 0)
obs_inj, *_ = obs_start.simulate(A({"redispatch": redisp}), 0)
obs_target, *_ = obs_start.simulate(A(act_s1) + A(act_s2) + A({"redispatch": redisp}), 0)

# no lines here: unit_act_observations are the sub actions, idls_subs in the same order
p_gst, alpha, betas = compute_flows_GST_from_unit_act_obs(
    [], obs_start, [obs1, obs2], [obs_inj], idls_subs=[id_sub1, id_sub2], return_betas=True)
print("alpha =", round(alpha, 4), " betas =", np.round(betas, 4))
print(f"GST       Max difference (MW): {np.max(np.abs(obs_target.p_or - p_gst))}")
p_est_only = compute_flows_GST_from_unit_act_obs(
    [], obs_start, [obs1, obs2], [], idls_subs=[id_sub1, id_sub2])   # EST = GST without injection
print(f"EST only  Max difference (MW): {np.max(np.abs(obs_target.p_or - p_est_only)):.4f}")
assert np.all(np.round(obs_target.p_or - p_gst, decimal_digit_precision) == 0.0)

alpha = -0.58  betas = [0.9182 0.6619]
GST       Max difference (MW): 2.9245902481989106e-06
EST only  Max difference (MW): 4.3152


### Mixed actions — a line disconnection **and** a node split, with an injection

The most general case combines several action *kinds* at once. Order the unitary
observations lines-first, subs-last, and set `idls = idls_lines + idls_subs`.

In [8]:
env.set_id(0); env.reset(); obs_start, *_ = env.step(A({}))
id_l, id_sub = 3, 5
act_line = {"set_line_status": [(id_l, -1)]}
act_sub = {"set_bus": {"substations_id": [(5, (1, 1, 2, 2, 1, 2, 2))]}}
obs_l, *_ = obs_start.simulate(A(act_line), 0)     # line action first
obs_s, *_ = obs_start.simulate(A(act_sub), 0)      # sub action last
obs_inj, *_ = obs_start.simulate(A({"redispatch": redisp}), 0)
obs_target, *_ = obs_start.simulate(A(act_line) + A(act_sub) + A({"redispatch": redisp}), 0)

p_gst, alpha, betas = compute_flows_GST_from_unit_act_obs(
    [id_l], obs_start, [obs_l, obs_s], [obs_inj], idls_subs=[id_sub], return_betas=True)
print("alpha =", round(alpha, 4), " betas =", np.round(betas, 4))
print(f"GST (line disconnection + node split + injection)  Max difference (MW): {np.max(np.abs(obs_target.p_or - p_gst))}")
assert np.all(np.round(obs_target.p_or - p_gst, decimal_digit_precision) == 0.0)

alpha = -0.9338  betas = [0.9938 0.9399]
GST (line disconnection + node split + injection)  Max difference (MW): 3.3632969760333253e-06


## Summary

* The GST reconstructs the combined **injection + topology** target flows from the
  precomputed unitary states only, with **no new power-flow solve** — matching the full
  DC solver to the notebook's 4-decimal accuracy.
* The topology states keep the EST weights; the injection states add in pure
  superposition; the injection enters the weights **only through the right-hand side**.
* `injection_observations = []` recovers the EST exactly, so the GST is a strict
  generalization. Substation split/merge actions are supported via `idls_subs` (node split = virtual-line disconnection, node merge = virtual-line reconnection), including mixed with line actions. For mixed action kinds, use
  `compute_flows_GST_from_unit_act_obs`; for same-kind combinations the
  `State.compute_flows_gst(...)` object API is also available.